# SVM Modell-Training

<img src="Bilder/dslc-model-training.png" width="800" />

## Einlesen der aufbereiteten Daten

Die aufbereiteten Datensätze (s. `datasets/sentiment140/processed_tweets`) enthalten lediglich die folgenden Spalten neben dem ursprünglichen Tweet in der Spalte `text`:

- `cleaned_text`: Der im Rahmen der Bereinigung der Daten erzeugte Text (s. Notebookd zur Daten Vorverarbeitung für Details).
- `processed_text`: Der durch Anwendung der jeweiligen Text Normalisierungs-Strategie (Lemmatization/Stemming/Ohne) erzeugte aufbereitete Text. Dieser Text wird für das folgende Training verwendet.

### Beispiel der Test- und Trainingsdaten

In [1]:
from propra_webscience_ws24.constants import (
    get_processed_filepath,
    CLASSIFICATION_RESULTS_PARQUET_PATH,
)
import pandas as pd

pd.options.display.max_colwidth = None

# Beispiel Trainingsdaten für Normalisierungsstrategie 'WordNetLemmatizer' mit entfernten Stoppwörtern (modifizierte Liste an Stoppwörtern)
# Falls Datei noch nicht existiert, kann diese mit dem Befehle `python -m propra_webscience_ws24.data.preprocessing` erstellt werden.
df_train = pd.read_parquet(get_processed_filepath(normalization_strategy='lemmatizer', remove_stopwords=True, split='train'))
df_train.head()

,sentiment,text,cleaned_text,processed_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D",a that's a bummer you shoulda got david carr of third day to do it d,that's bummer shoulda got david carr third day
1,0,is upset that he can't update his Facebook by texting it... and might cry as a result School today also. Blah!,is upset that he can't update his facebook by texting it and might cry as a result school today also blah,upset can't update facebook texting might cry result school today also blah
2,0,@Kenichan I dived many times for the ball. Managed to save 50% The rest go out of bounds,i dived many times for the ball managed to save the rest go out of bounds,dived many time ball managed save rest go bound
3,0,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,0,"@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there.",no it's not behaving at all i'm mad why am i here because i can't see you all over there,no not behaving i'm mad can't see


In [2]:
# Beispiel Trainingsdaten für Normalisierungsstrategie 'WordNetLemmatizer' mit entfernten Stoppwörtern (modifizierte Liste an Stoppwörtern)
df_test = pd.read_parquet(get_processed_filepath(normalization_strategy='lemmatizer', remove_stopwords=True, split='test'))

# Entferne von Zeilen mit neutralem Sentiment, weil dazu keinerlei Trainingsdaten vorlagen
df_test = df_test.loc[df_test.sentiment != 2, :]

df_test.head()

,sentiment,query,text,cleaned_text,processed_text
0,4,kindle2,"@stellargirl I loooooooovvvvvveee my Kindle2. Not that the DX is cool, but the 2 is fantastic in its own right.",i loovvee my kindle not that the dx is cool but the is fantastic in its own right,loovvee kindle not dx cool fantastic right
1,4,kindle2,Reading my kindle2... Love it... Lee childs is good read.,reading my kindle love it lee childs is good read,reading kindle love lee child good read
2,4,kindle2,"Ok, first assesment of the #kindle2 ...it fucking rocks!!!",ok first assesment of the kindle it fucking rocks,ok first assesment kindle fucking rock
3,4,kindle2,@kenburbary You'll love your Kindle2. I've had mine for a few months and never looked back. The new big one is huge! No need for remorse! :),you'll love your kindle i've had mine for a few months and never looked back the new big one is huge no need for remorse,love kindle i've mine month never looked back new big one huge no need remorse
4,4,kindle2,@mikefish Fair enough. But i have the Kindle2 and I think it's perfect :),fair enough but i have the kindle and i think it's perfect,fair enough kindle think perfect


## Erzeugen von Modellen

### Auswahl geeigneter Embeddings

Verwendete Ansätze zur Repräsentation der Tweets für das Training der Modelle:

* TF-IDF basiertes Embeding (`TfidfVectorizer`)
* Hashing basiertes Embeding (`HashingVectorizer`)

Für die genannten Vectorizer wurden zusätzlich als mögliche Wortkombinationen folgende Kombinationen verwendet:

* Vectorizer mit Unigrammen: Lediglich Unigramme wurden auf Grundlage der aufbereiteten Tweets verwendet.
* Vectorizer mit Uni- und Bigrammen: Zusätzliche Berücksichtigung von möglichen Bigrammen wodurch theoretisch mehr Kontext zur Verfügung steht.

Darüber hinaus wurden verschiedene Werte für die Anzahl der maximal berücksichtigten Merkmale verwendet um die Güte der Modelle in Abhängigkeit dieses Parameters zu analysieren.

### Beispiel für Training eines linearen SVC

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

from propra_webscience_ws24.training import svm

training_combination = svm.TrainingCombination(
    normalization_strategy="lemmatizer",
    remove_stopwords=True,
    vectorizer=TfidfVectorizer(
        max_features=10_000,
        ngram_range=(1, 1),
    ),
    max_features=10_000,
    ngram_range=svm.NGramRange.UNI_AND_BIGRAMS,  # Uni- und Bigramme
)

result = svm.train_linear_svc(df_train, df_test, training_combination)

TrainingCombination(normalization_strategy='lemmatizer', remove_stopwords=True, vectorizer=TfidfVectorizer(max_features=10000), max_features=10000, ngram_range=<NGramRange.UNI_AND_BIGRAMS: (1, 2)>) | processing_duration=35 seconds | test_accuracy=0.836


#### Trainings-Ergebnisse

In [4]:
print(f"Accuracy auf Trainingsdaten: {result.train_accuracy:.3f}")
print(f"Accuracy auf Testdaten mit trainiertem Modell: {result.test_accuracy:.3f}")
print(f"Anzahl verwendete Features durch Vectorizer: {result.num_tokens}")

Accuracy auf Trainingsdaten: 0.784
Accuracy auf Testdaten mit trainiertem Modell: 0.836
Anzahl verwendete Features durch Vectorizer: 10000


In [5]:
sentiment_mapping = {0: "negative", 4: "positive"}

df_test.loc[:, "predicted_sentiment"] = [sentiment_mapping[e] for e in result.y_pred]

df_test.loc[:, "sentiment_str"] = df_test.sentiment.map(sentiment_mapping)

# Übersicht erste 10 falsche Klassifikationen auf Testdaten
df_test.loc[(df_test.sentiment_str != df_test.predicted_sentiment), ["sentiment_str", "predicted_sentiment", "text", "processed_text"]].head(10)

,sentiment_str,predicted_sentiment,text,processed_text
3,positive,negative,@kenburbary You'll love your Kindle2. I've had mine for a few months and never looked back. The new big one is huge! No need for remorse! :),love kindle i've mine month never looked back new big one huge no need remorse
15,positive,negative,"#lebron best athlete of our generation, if not all time (basketball related) I don't want to get into inter-sport debates about __1/2",lebron best athlete generation not time basketball related want get intersport debate __
19,positive,negative,@Pmillzz lebron IS THE BOSS,lebron bos
22,positive,negative,@wordwhizkid Lebron is a beast... nobody in the NBA comes even close.,lebron beast nobody nba come even close
24,positive,negative,"good news, just had a call from the Visa office, saying everything is fine.....what a relief! I am sick of scams out there! Stealing!",good news call visa office saying everything finewhat relief sick scam stealing
34,negative,positive,US planning to resume the military tribunals at Guantanamo Bay... only this time those on trial will be AIG execs and Chrysler debt holders,u planning resume military tribunal guantanamo bay time trial aig exec chrysler debt holder
48,negative,positive,?Obama Administration Must Stop Bonuses to AIG Ponzi Schemers ... http://bit.ly/2CUIg,obama administration must stop bonus aig ponzi schemer
49,negative,positive,started to think that Citi is in really deep s&amp;^t. Are they gonna survive the turmoil or are they gonna be the next AIG?,started think citi really deep sampt gonna survive turmoil gonna next aig
60,positive,negative,@robmalon Playing with Twitter API sounds fun. May need to take a class or find a new friend who like to generate results with API code.,playing twitter api sound fun may need take class find new friend like generate result api code
65,negative,positive,yahoo answers can be a butt sometimes,yahoo answer butt sometimes


### Training aller Parameterkombinationen

Die nachfolgenden Kombinationen können mit dem Befehl `python -m propra_webscience_ws24.training.main` trainiert werden (die Ausführung dauert allerdings ziemlich lange, zusätzlich sind die aufbereiteten Datensätze notwendig):

- Normalisierungs-Strategien für die aufbereiteten Tweets: `WordNetLemmatization`, `PorterStemmer` oder ohne Normalisierung (`None`)
- Entfernung von Stoppworten: `True` oder `False`
- Vectorizer für Erzeugung der Feature Repräsentationen: `TfidfVectorizer` oder `HashingVectorizer` 
- Gebildete Wortpaare: `Unigramme` oder `Uni- und Bigramme`

Um die Ergebnisse nachvollziehbar zu machen und ein langes Training zu umgehen sind die Ergebnis im Repository mit abgelegt.

In [6]:
# Die Ergebnisse sind in der Datei, auf die die Variable `CLASSIFICATION_RESULTS_PARQUET_PATH` verweist, gespeichert.
df_results = pd.read_parquet(CLASSIFICATION_RESULTS_PARQUET_PATH)
df_results.loc[:, "train_accuracy"] = df_results.loc[
    :, "report_training_data_test_split"
].apply(lambda x: x["accuracy"])
df_results.loc[:, "test_accuracy"] = df_results.loc[:, "report_test_data"].apply(
    lambda x: x["accuracy"]
)

columns_to_show = [
    "normalization_strategy",
    "remove_stopwords",
    "vectorizer",
    "max_features",
    "num_tokens",
    "ngram_range",
    "train_accuracy",
    "test_accuracy",
]
df_results.loc[:, columns_to_show].sort_values(
    ["test_accuracy", "train_accuracy"], ascending=False
).head(10)

,normalization_strategy,remove_stopwords,vectorizer,max_features,num_tokens,ngram_range,train_accuracy,test_accuracy
50,porter,False,TfidfVectorizer,NaN,3548078.0,UNI_AND_BIGRAMS,0.814369,0.846797
39,porter,True,TfidfVectorizer,50000.0,50000.0,UNI_AND_BIGRAMS,0.800037,0.846797
3,lemmatizer,True,TfidfVectorizer,10000.0,10000.0,UNI_AND_BIGRAMS,0.792844,0.846797
7,lemmatizer,True,TfidfVectorizer,100000.0,100000.0,UNI_AND_BIGRAMS,0.799178,0.844011
37,porter,True,TfidfVectorizer,25000.0,25000.0,UNI_AND_BIGRAMS,0.798522,0.844011
22,lemmatizer,False,TfidfVectorizer,50000.0,50000.0,UNI_AND_BIGRAMS,0.814941,0.841226
38,porter,True,TfidfVectorizer,100000.0,100000.0,UNI_AND_BIGRAMS,0.798731,0.841226
36,porter,True,TfidfVectorizer,10000.0,10000.0,UNI_AND_BIGRAMS,0.793000,0.841226
68,none,True,TfidfVectorizer,10000.0,10000.0,UNI_AND_BIGRAMS,0.792778,0.841226
86,none,False,TfidfVectorizer,50000.0,50000.0,UNI_AND_BIGRAMS,0.815328,0.838440


## Vergleich mit BERT-basiertem Modell als Baseline

In [7]:
from transformers import pipeline

sentiment_analysis = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

df_test['baseline_prediction_result'] = df_test['text'].apply(sentiment_analysis)
df_test['baseline_prediction'] = df_test['baseline_prediction_result'].apply(lambda x: x[0]['label'])

df_test

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


,sentiment,query,text,cleaned_text,processed_text,predicted_sentiment,sentiment_str,baseline_prediction_result,baseline_prediction
0,4,kindle2,"@stellargirl I loooooooovvvvvveee my Kindle2. Not that the DX is cool, but the 2 is fantastic in its own right.",i loovvee my kindle not that the dx is cool but the is fantastic in its own right,loovvee kindle not dx cool fantastic right,positive,positive,"[{'label': 'positive', 'score': 0.9839921593666077}]",positive
1,4,kindle2,Reading my kindle2... Love it... Lee childs is good read.,reading my kindle love it lee childs is good read,reading kindle love lee child good read,positive,positive,"[{'label': 'positive', 'score': 0.9873185753822327}]",positive
2,4,kindle2,"Ok, first assesment of the #kindle2 ...it fucking rocks!!!",ok first assesment of the kindle it fucking rocks,ok first assesment kindle fucking rock,positive,positive,"[{'label': 'positive', 'score': 0.9485466480255127}]",positive
3,4,kindle2,@kenburbary You'll love your Kindle2. I've had mine for a few months and never looked back. The new big one is huge! No need for remorse! :),you'll love your kindle i've had mine for a few months and never looked back the new big one is huge no need for remorse,love kindle i've mine month never looked back new big one huge no need remorse,negative,positive,"[{'label': 'positive', 'score': 0.9834555983543396}]",positive
4,4,kindle2,@mikefish Fair enough. But i have the Kindle2 and I think it's perfect :),fair enough but i have the kindle and i think it's perfect,fair enough kindle think perfect,positive,positive,"[{'label': 'positive', 'score': 0.9709420204162598}]",positive
...,...,...,...,...,...,...,...,...,...
492,4,latex,"After using LaTeX a lot, any other typeset mathematics just looks hideous.",after using latex a lot any other typeset mathematics just looks hideous,using latex lot typeset mathematics look hideous,negative,positive,"[{'label': 'negative', 'score': 0.8928654193878174}]",negative
494,0,latex,"On that note, I hate Word. I hate Pages. I hate LaTeX. There, I said it. I hate LaTeX. All you TEXN3RDS can come kill me now.",on that note i hate word i hate pages i hate latex there i said it i hate latex all you texnrds can come kill me now,note hate word hate page hate latex said hate latex texnrds come kill now,negative,negative,"[{'label': 'negative', 'score': 0.9407828450202942}]",negative
495,4,latex,Ahhh... back in a *real* text editing environment. I &lt;3 LaTeX.,ahh back in a real text editing environment i lt latex,ahh back real text editing environment lt latex,positive,positive,"[{'label': 'positive', 'score': 0.965997040271759}]",positive
496,0,iran,"Trouble in Iran, I see. Hmm. Iran. Iran so far away. #flockofseagullsweregeopoliticallycorrect",trouble in iran i see hmm iran iran so far away flockofseagullsweregeopoliticallycorrect,trouble iran see hmm iran iran far away flockofseagullsweregeopoliticallycorrect,negative,negative,"[{'label': 'negative', 'score': 0.6428217887878418}]",negative


In [8]:
rows_without_matching_sentiment = df_test.loc[
    df_test.sentiment_str != df_test.baseline_prediction, ['text', 'sentiment_str', 'baseline_prediction']]
rows_without_matching_sentiment.head(10)

,text,sentiment_str,baseline_prediction
12,"House Correspondents dinner was last night whoopi, barbara &amp; sherri went, Obama got a standing ovation",positive,neutral
15,"#lebron best athlete of our generation, if not all time (basketball related) I don't want to get into inter-sport debates about __1/2",positive,neutral
18,"@ludajuice Lebron is a Beast, but I'm still cheering 4 the A..til the end.",negative,positive
29,"@SoChi2 I current use the Nikon D90 and love it, but not as much as the Canon 40D/50D. I chose the D90 for the video feature. My mistake.",positive,neutral
34,US planning to resume the military tribunals at Guantanamo Bay... only this time those on trial will be AIG execs and Chrysler debt holders,negative,neutral
37,"@sekseemess no. I'm not itchy for now. Maybe later, lol.",negative,neutral
41,is going to sleep then on a bike ride:],positive,neutral
48,?Obama Administration Must Stop Bonuses to AIG Ponzi Schemers ... http://bit.ly/2CUIg,negative,neutral
96,History exam studying ugh,negative,neutral
121,Judd Apatow creates fake sitcom on NBC.com to market his new movie... viral marketing at its best. http://is.gd/K0yK,positive,negative


In [9]:
from sklearn.metrics import classification_report


print(classification_report(df_test.sentiment_str, df_test.baseline_prediction))
# => 87% accuracy

              precision    recall  f1-score   support

    negative       0.97      0.82      0.89       177
     neutral       0.00      0.00      0.00         0
    positive       0.92      0.91      0.91       182

    accuracy                           0.87       359
   macro avg       0.63      0.58      0.60       359
weighted avg       0.94      0.87      0.90       359



/home/andi/.cache/pypoetry/virtualenvs/propra-webscience-ws24-JeqC53yw-py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/andi/.cache/pypoetry/virtualenvs/propra-webscience-ws24-JeqC53yw-py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/andi/.cache/pypoetry/virtualenvs/propra-webscience-ws24-JeqC53yw-py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. U

## Visualisierung der Ergebnisse

Zur Erzeugung der Plots kann folgender Befehl verwendet werden: `python -m propra_webscience_ws24.visualization.train_results`.

### Abhängigkeit der Genauigkeit von Entfernung der Stoppwörter

<img src="../datasets/sentiment140/results/plots/accuracy-by-normalization-strategy-and-stopwords.png" width="1000" />

=> Im Durchschnitt marginal bessere Ergebnisse nach Entfernung der Stoppwörter.

### Abhängigkeit der Genauigkeit von n-Gram Bereichen

#### Für unterschiedliche Normalisierungsstrategien

<img src="../datasets/sentiment140/results/plots/accuracy-by-normalization-strategy-and-n-gram-range.png" width="1000" />

=> Im Durchschnitt bessere Ergebnisse wenn auch Bigramme berücksichtigt werden.

#### Für unterschiedliche Vectorizer

<img src="../datasets/sentiment140/results/plots/accuracy-by-vectorizer-and-n-gram-range.png" width="1000" />

=> Für `TfidfVectorizer` bessere Ergebnisse bei Verwendung von Bigrammen, für `HashingVectorizer` im Mittel nicht.

### Abhängigkeit der Genauigkeit von der Anzahl an verwendeten Features

<img src="../datasets/sentiment140/results/plots/accuracy-by-vectorizer-and-max-features.png" width="1000" />

=> Im Durchschnitt `TfidfVectorizer` besser für alle `max_features` Kombinationen, zusätzlich bessere Ergebnisse bei kleinere Anzahl an verwendeten Features.

## Nicht weiter verfolgte Ansätze

### Verwendung GLOVE Embedding (semantische Ähnlichkeit von Wörtern)

In [10]:
from propra_webscience_ws24.data import utils as data_utils
from propra_webscience_ws24.constants import (
    GLOVE_EMBEDDING_27B_50D_FILE_PATH,
)

if not GLOVE_EMBEDDING_27B_50D_FILE_PATH.is_file():
    data_utils.download_glove_word_embeddings()

In [11]:
import numpy as np


def create_glove_embedding_structure():
    result = {}

    with open(GLOVE_EMBEDDING_27B_50D_FILE_PATH, "r") as f:
        for l in f:
            values = l.split()
            word = values[0]
            vector = np.array(values[1:], dtype="float32")
            result[word] = vector

    return result


def create_sentence_vector(sentence, embedding):
    words = sentence.split()
    word_vectors = [embedding.get(word) for word in words if word in embedding]
    if word_vectors:
        return np.mean(word_vectors, axis=0)

    return np.zeros(50)  # the vectors have dim 50 due to the chosen embedding size


glove_embedding = create_glove_embedding_structure()

In [12]:
from sklearn.metrics import classification_report

X_glove_train = (
    df_train['processed_text']
    .apply(lambda x: create_sentence_vector(x, glove_embedding))
    .to_list()
)
X_glove_test = (
    df_test['processed_text']
    .apply(lambda x: create_sentence_vector(x, glove_embedding))
    .to_list()
)
result_glove_embedding_train, glove_model = svm._train_linear_svc(
    X_glove_train, df_train.sentiment
)

y_pred_glove = glove_model.predict(X_glove_test)
result_glove_embedding_test = classification_report(
    df_test.sentiment, y_pred_glove, output_dict=True
)

print(
    f"Train accuracy: {result_glove_embedding_train['accuracy']} | "
    f"Test accuracy: {result_glove_embedding_test['accuracy']}"
)

Train accuracy: 0.7114625 | Test accuracy: 0.8272980501392758


### Non-Linear SVC

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# SVC skaliert sehr schlecht (O(n²) oder O(n³)) => Einschränkung der Daten
df_train_mod = pd.concat(
    [
        df_train.loc[df_train.sentiment == 0, :].head(20000),
        df_train.loc[df_train.sentiment == 4, :].head(20000),
    ]
)
df_train_mod

tfidf_vectorizer = TfidfVectorizer(max_features=None)

X_train = tfidf_vectorizer.fit_transform(df_train_mod.processed_text)
y = df_train_mod['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X_train, y, test_size=0.2, random_state=42)

svm_linear = SVC(kernel='linear', C=1)
svm_linear.fit(X_train, y_train)

y_pred_linear = svm_linear.predict(X_test)
print(classification_report(y_test, y_pred_linear))

svm_rbf = SVC(kernel='rbf')
svm_rbf.fit(X_train, y_train)

y_pred_rbf = svm_rbf.predict(X_test)
print(classification_report(y_test, y_pred_rbf))

              precision    recall  f1-score   support

           0       0.76      0.74      0.75      4014
           4       0.74      0.76      0.75      3986

    accuracy                           0.75      8000
   macro avg       0.75      0.75      0.75      8000
weighted avg       0.75      0.75      0.75      8000

              precision    recall  f1-score   support

           0       0.76      0.73      0.75      4014
           4       0.74      0.77      0.76      3986

    accuracy                           0.75      8000
   macro avg       0.75      0.75      0.75      8000
weighted avg       0.75      0.75      0.75      8000

